# Phase 3 — Out-of-Time Test & Statistical Testing

**MSc Credit Risk Project — Colab handover (3 of 6)**

1. Trains the 9 configurations on the **full** 2018-2019 window and evaluates once on
   the untouched 2020 cohort (the out-of-time benchmark).
2. Runs the **16-comparison statistical battery** (paired t-tests on the 15 paired CV
   observations per comparison, plus the OOT champion check).
3. Produces the evaluation figures: ROC curves, PR curves, confusion matrices,
   cost curves, and the significance heatmap.

**Checkpoints:** `reports/tables/oot_results.csv`, `models/oot_probas.parquet`, figures.
**Runtime:** ~10 minutes.


In [ ]:
# ============================================================
# SETUP: mount Google Drive and locate the project folder
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

# AUTO-DETECT the project root (folder containing src/ and run_experiment.py).
# If auto-detection fails, set PROJECT_ROOT manually, e.g.:
#   PROJECT_ROOT = Path('/content/drive/MyDrive/credit-risk-project - Copy')
PROJECT_ROOT = None
for candidate in Path('/content/drive/MyDrive').rglob('run_experiment.py'):
    if (candidate.parent / 'src').is_dir():
        PROJECT_ROOT = candidate.parent
        break
assert PROJECT_ROOT is not None, "Could not find the project folder on Drive - set PROJECT_ROOT manually"

os.chdir(PROJECT_ROOT)
import sys
sys.path.insert(0, str(PROJECT_ROOT))
# Ensure expected directories exist (log files are opened at import time)
for _d in ('logs', 'data/raw', 'data/processed', 'models',
           'reports/tables', 'reports/figures', 'outputs/eda'):
    os.makedirs(PROJECT_ROOT / _d, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print("Working directory set. All outputs are saved here (persistent on Drive).")


In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

from src.config import TABLES_DIR

if (TABLES_DIR / "oot_results.csv").exists():
    print("OOT checkpoint found - SKIPPING OOT track.")
else:
    from run_experiment import load_and_split, run_oot_track
    df_train, df_test = load_and_split()
    df_oot = run_oot_track(df_train, df_test)
    df_oot.to_csv(TABLES_DIR / "oot_results.csv", index=False)
    print("OOT track complete.")


In [ ]:
# ---- Display the OOT results (Table 4.2 of the dissertation) ----
import pandas as pd
df_oot = pd.read_csv(TABLES_DIR / "oot_results.csv")
print("Out-of-time (2020) results, sorted by PR-AUC:")
df_oot.sort_values('pr_auc', ascending=False).round(4)


In [ ]:
# Statistical battery + evaluation plots (identical to run_analysis.py steps 1-2)
from src.utils.stats_testing import run_all_tests
from src.config import FIGURES_DIR, MODELS_DIR

df_stats = run_all_tests(
    cv_path=TABLES_DIR / "cv_fold_results.csv",
    out_path=TABLES_DIR / "statistical_significance.csv",
    figure_path=FIGURES_DIR / "statistical_significance_heatmap.png",
)
print(f"Significant at alpha=0.05: {int(df_stats['significant_alpha_0.05'].sum())} / {len(df_stats)}")
df_stats.round(6)


In [ ]:
# ---- Display the evaluation figures inline ----
from IPython.display import Image, display
from src.models.plots import plot_confusion_grid, plot_cost_curves, plot_pr_curves, plot_roc_curves
probas_path = MODELS_DIR / "oot_probas.parquet"
plot_roc_curves(probas_path, FIGURES_DIR / "roc_curves_oot.png")
plot_pr_curves(probas_path, FIGURES_DIR / "pr_curves_oot.png")
plot_confusion_grid(probas_path, FIGURES_DIR / "confusion_matrices_oot.png")
plot_cost_curves(probas_path, FIGURES_DIR / "cost_curves_oot.png")
print("Evaluation figures saved to reports/figures/\n")

for name in ["roc_curves_oot.png", "pr_curves_oot.png",
             "confusion_matrices_oot.png", "cost_curves_oot.png",
             "statistical_significance_heatmap.png"]:
    display(Image(str(FIGURES_DIR / name), width=700))


In [ ]:
# ---- Phase 3 verification against dissertation Table 4.2 / 4.3 ----
champ = df_oot[df_oot['display_name'] == 'XGBoost (Baseline)'].iloc[0]
assert abs(champ['roc_auc'] - 0.7669) < 0.002, champ['roc_auc']
assert abs(champ['pr_auc'] - 0.2513) < 0.005, champ['pr_auc']

cs = df_oot[df_oot['display_name'] == 'XGBoost (Cost-Sensitive)'].iloc[0]
assert abs(cs['recall'] - 0.657) < 0.01, cs['recall']

df_stats = pd.read_csv(TABLES_DIR / "statistical_significance.csv")
assert int(df_stats['significant_alpha_0.05'].sum()) == 15
print("PHASE 3 CHECKPOINT OK: OOT ROC-AUC 0.7669, PR-AUC 0.2513, cost-sensitive recall 0.657, 15/16 significant")
